---

# 스프린트미션16 4팀_김명환

## 1. 기본 라이브러리 / 함수
### 1.1. 라이브러리

In [1]:
import importlib
import sys
import subprocess

def install_if_missing(package_name, module_name=None, index_url=None):
    # module_name이 없으면 package_name을 그대로 사용
    module_name = module_name or package_name.replace("-", "_")

    if importlib.util.find_spec(module_name) is None:
        print(f"{package_name} 설치 중...")
        cmd = [sys.executable, "-m", "pip", "install"]
        if index_url:
            cmd += ["--index-url", index_url]
        cmd.append(package_name)
        subprocess.check_call(cmd)
    else:
        print(f"{package_name} 이미 설치되어 있음.")

# 사용 예시
install_if_missing("helper-plot-hangul", "helper_plot_hangul", "https://test.pypi.org/simple/")
install_if_missing("helper-utils", "helper_utils", "https://test.pypi.org/simple/")


helper-plot-hangul 이미 설치되어 있음.
helper-utils 이미 설치되어 있음.


In [2]:
# import importlib
# from helper_plot_hangul import helper_plot_hangul
# importlib.reload(helper_plot_hangul)

# import helper_utils.helper_logger as helper_logger
# importlib.reload(helper_logger)

# import helper_utils.helper_utils_colab as helper_utils_colab
# importlib.reload(helper_utils_colab)

from helper_plot_hangul import *
from helper_utils.helper_logger import *
from helper_utils.helper_utils_colab import *
from helper_utils.helper_utils_print import *
from helper_utils.helper_pandas import *


In [3]:
# 기본 라이브러리

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import average_precision_score

# --- 기타 라이브러리 ---
from PIL import Image
from PIL import ImageFilter
from PIL import ImageDraw
import albumentations as A
import IPython.display
#from tqdm import tqdm
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss
from collections import OrderedDict

# --- 기타 ---
import re
import os
import sys
import copy
import json
import math
import random
import yaml
import shutil
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from datetime import datetime
from datetime import timezone, timedelta
import pytz
__kst = pytz.timezone('Asia/Seoul')

# GPU 설정
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

  # 재현 가능한 결과를 위해
np.random.seed(42)
torch.manual_seed(42)
if __device == 'cuda':
    torch.cuda.manual_seed_all(42)

print(f"라이브러리 로드 완료 사용장치:{__device}")

라이브러리 로드 완료 사용장치:cpu


### > 설정 < 플레그

In [4]:
DEBUG_ON = False if IS_COLAB else True
DEBUG_ON = False
TRAIN_ON = False
logger.info(f"IS_COLAB={IS_COLAB}")
logger.info(f"DEBUG_ON={DEBUG_ON}")


2025-12-07 19:52:15 I [helper_pandas:4] - IS_COLAB=False
2025-12-07 19:52:15 I [helper_pandas:5] - DEBUG_ON=False
2025-12-07 19:52:15 I [helper_pandas:5] - DEBUG_ON=False


## 2. 데이터 로드

- list.txt 파싱

In [5]:
root_cache_path = my_cache()
root_my_driver = my_driver()

logger.debug(f"root_cache_path: {root_cache_path}")
logger.debug(f"root_my_driver: {root_my_driver}")

### Yolo DataSet

In [6]:
import os, sys
import importlib
sys.path.insert(0, os.getcwd())

# 기존 모듈 완전 제거
if 'yolo_eval' in sys.modules:
    del sys.modules['yolo_eval']
    
# 하위 모듈도 제거
for key in list(sys.modules.keys()):
    if key.startswith('yolo_eval.'):
        del sys.modules[key]

# 새로 임포트
from yolo_eval import *

logger.info("YOLOEvaluator 클래스 로드 완료")

2025-12-07 19:52:15 I [helper_logger:58] - EvaluationMetrics 클래스 로드 완료
2025-12-07 19:52:15 I [helper_logger:24] - PredictionResult 클래스 로드 완료
2025-12-07 19:52:15 I [helper_logger:24] - PredictionResult 클래스 로드 완료
2025-12-07 19:52:15 I [helper_logger:637] - QuantizedModelWrapper 모듈 로드 완료
2025-12-07 19:52:15 I [helper_logger:449] - YOLOEvaluator 클래스 로드 완료
2025-12-07 19:52:15 I [helper_logger:255] - YOLOEvaluationPipeline 클래스 로드 완료
2025-12-07 19:52:15 I [helper_logger:637] - QuantizedModelWrapper 모듈 로드 완료
2025-12-07 19:52:15 I [helper_logger:449] - YOLOEvaluator 클래스 로드 완료
2025-12-07 19:52:15 I [helper_logger:255] - YOLOEvaluationPipeline 클래스 로드 완료
2025-12-07 19:52:15 I [helper_pandas:17] - YOLOEvaluator 클래스 로드 완료
2025-12-07 19:52:15 I [helper_pandas:17] - YOLOEvaluator 클래스 로드 완료


In [7]:
logger.setLevel(logging.INFO)
yolo_dataset_path = my_cache_path("yolo", "the-oxfordiiit-pet-dataset_eval")
yaml_path, train_df, valid_df, test_df, validation_results = oxfordiit_pet_to_yolo(max_samples_per_split=(30, 20, 10),
                                                                                   label_mode="species",
                                                                                   output_dir=Path(yolo_dataset_path))
logger.setLevel(logging.DEBUG)

Processing test: 100%|██████████| 10/10 [00:00<00:00, 1867.04it/s]
2025-12-07 19:52:16,166 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\train\Abyssinian_104.txt
Processing test: 100%|██████████| 10/10 [00:00<00:00, 1867.04it/s]
2025-12-07 19:52:16,166 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\train\Abyssinian_104.txt
2025-12-07 19:52:16,211 - yolo_eval.oxfordiiit_pet_dataset - INFO - ==================================================
2025-12-07 19:52:16,212 - yolo_eval.oxfordiiit_pet_dataset - INFO - 변환 완료!
2025-12-07 19:52:16,212 - yolo_eval.oxfordiiit_pet_dataset - INFO - YAML 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\data.yaml
2025-12-07 19:52:16,213 - yolo_eval.oxfordiiit_pet_dataset - INFO - 검증 결과: {'train_samples': 29, 'valid_samples': 20, 'test_samples': 10, 'total_samples': 59, 'num_classes': 1, 'class_dist

## 3. 평가

In [8]:
EVAL_MODEL = {}
EVAL_MODEL['YOLOv8m_baseline'] = False
EVAL_MODEL['YOLOv8m_int8_openvino'] = False
EVAL_MODEL['YOLOv8m_ONNX_FP32'] = True
EVAL_MODEL['YOLOv8m_ONNX_FP16'] = False
EVAL_MODEL['YOLOv8m_ONNX_int8_qdq'] = True


In [9]:
logger.info(f"yaml_path: {yaml_path}")
pipeline = YOLOEvaluationPipeline(
    yaml_path=yaml_path,
    device=str(__device)
)


2025-12-07 19:52:16 I [helper_pandas:1] - yaml_path: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\data.yaml


In [10]:
logger.setLevel(logging.INFO)
yolov8m_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005')
logger.setLevel(logging.DEBUG)


2025-12-07 19:52:16 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005


In [11]:
# 기본
if EVAL_MODEL['YOLOv8m_baseline']:
    yolov8m_best_path = my_driver_path(yolov8m_path, 'weights', 'best.pt', create=False)
    logger.info(f"yolov8m_best_path: {yolov8m_best_path}")
    pipeline.add_model(
        model_path=yolov8m_best_path,
        model_name="YOLOv8m_baseline",
        verbose=False,
    )

In [12]:
# Yolo 8 int 8양자화
if EVAL_MODEL['YOLOv8m_int8_openvino']:
    output_openvino_int8_path = my_driver_path(yolov8m_path, 'weights', 'best_int8_openvino_model', create=False)
    logger.info(f"output_openvino_int8_path: {output_openvino_int8_path}")

    # OpenVINO 모델은 디렉토리 경로를 전달해야 합니다
    # Ultralytics는 '_openvino_model' suffix로 OpenVINO 형식을 감지합니다
    pipeline.add_model(
        model_path=output_openvino_int8_path,  # 디렉토리 경로 (수정됨)
        model_name="YOLOv8m_int8_openvino",
        verbose=False,
        model_type='openvino',
    )


In [13]:
# YOLOv8m_ONNX_FP32
if EVAL_MODEL['YOLOv8m_ONNX_FP32']:
    yolov8m_onnx_fp32_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp32', 'yolov8m_fp32.onnx', create=False)
    pipeline.add_model(
        model_path=yolov8m_onnx_fp32_path,
        model_name="YOLOv8m_ONNX_FP32",
        verbose=False,
    )

2025-12-07 19:52:16 I [helper_logger:95] - 모델 추가: YOLOv8m_ONNX_FP32 (타입: yolo_pt)
2025-12-07 19:52:16 I [helper_logger:75] - 모델 로딩 중 (타입: yolo_pt): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
2025-12-07 19:52:16 I [helper_logger:190] - 모델 로드 완료: YOLOv8m_ONNX_FP32
2025-12-07 19:52:16 I [helper_logger:201] - 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval
2025-12-07 19:52:16 I [helper_logger:202] - 테스트 이미지: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test
2025-12-07 19:52:16 I [helper_logger:75] - 모델 로딩 중 (타입: yolo_pt): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for you

In [14]:
# YOLOv8m_ONNX_FP16
if EVAL_MODEL['YOLOv8m_ONNX_FP16']:
    yolov8m_onnx_fp16_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp16', 'yolov8m_fp16.onnx', create=False)
    pipeline.add_model(
        model_path=yolov8m_onnx_fp16_path,
        model_name="YOLOv8m_ONNX_FP16",
        verbose=False,
    )


In [15]:
# YOLOv8m_ONNX_int8
if EVAL_MODEL['YOLOv8m_ONNX_int8_qdq']:
    yolov8m_onnx_int8_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_int8_qdq', 'yolov8m_int8_qdq.onnx', create=False)
    pipeline.add_model(
        model_path=yolov8m_onnx_int8_path,
        model_name="YOLOv8m_ONNX_int8_qdq",
        verbose=False,
        model_type='onnx',
    )


2025-12-07 19:52:16 I [helper_logger:95] - 모델 추가: YOLOv8m_ONNX_int8_qdq (타입: onnx)
2025-12-07 19:52:16 I [helper_logger:75] - 모델 로딩 중 (타입: onnx): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_int8_qdq\yolov8m_int8_qdq.onnx
2025-12-07 19:52:16 I [helper_logger:83] - ONNX 모델을 QuantizedModelWrapper로 로딩
2025-12-07 19:52:16 I [helper_logger:86] - ONNX 모델 크기: 25.12 MB
2025-12-07 19:52:16 I [helper_logger:75] - 모델 로딩 중 (타입: onnx): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_int8_qdq\yolov8m_int8_qdq.onnx
2025-12-07 19:52:16 I [helper_logger:83] - ONNX 모델을 QuantizedModelWrapper로 로딩
2025-12-07 19:52:16 I [helper_logger:86] - ONNX 모델 크기: 25.12 MB
2025-12-07 19:52:16 I [helper_logger:181] - ONNX 모델 로딩: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_int8_qdq\yolov8m_int8_qdq.onnx
2025-12-07 19:52:16 I [helper_logger:204] - ONNX 입력: images, shape=[1, 3, 640, 640]
2025-12-07 19:52:16 I [helper_logg

In [16]:
# yolov8m_best_path = my_driver_path(yolov8m_path, 'weights', 'best.pt', create=False)
# evaluator = YOLOEvaluator(
#     model_path=yolov8m_best_path,
#     yaml_path=yaml_path,
#     model_name="yolov8m_fp32",
#     device=str(__device),
#     verbose=False,
#     model_type="yolo_pt",
# )
# evaluator.model


In [17]:

# yolov8m_onnx_fp32_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp32', 'yolov8m_fp32.onnx', create=False)
# evaluator = YOLOEvaluator(
#     model_path=yolov8m_onnx_fp32_path,
#     yaml_path=yaml_path,
#     model_name="yolov8m_fp32",
#     device=str(__device),
#     verbose=False,
#     model_type="onnx",
# )
# evaluator.model


In [ ]:
# 전체 평가 실행
yolov8m_result_path = my_driver_path(yolov8m_path, 'weights', 'result', create=True)
results = pipeline.run_full_evaluation(
    val_kwargs={
        'project': yolov8m_result_path,
        'split': 'test',
        'imgsz': 640,
        'batch': 1,      # ONNX 고정 shape 문제로 배치 크기 1로 축소
        'conf': 0.001,
        'iou': 0.45,
        'rect': False    # rectangular inference 비활성화 (동일 크기 강제)
    },
    pred_conf=0.001,
    pred_imgsz=640       # 예측 시에도 640x640 강제
)
# results = pipeline.run_full_evaluation(
#     val_kwargs={
#         'project': yolov8m_result_path,  # custom_results/ 디렉터리에
#         # 'conf': mission_16_yolo_yaml,
#         # 'name': 'baseline_test',      # baseline_test/ 하위 폴더로 저장
#         'split': 'test',
#         'imgsz': 640,
#         'batch': 16,
#         'conf': 0.25,
#         'iou': 0.75
#     },
#     # pred_max_images=100,
#     pred_conf=0.25
# )


2025-12-07 19:52:17 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\result
모델 검증 시작

[YOLOv8m_ONNX_FP32] 검증 중...
2025-12-07 19:52:17 I [helper_logger:251] - 모델 검증 시작: split=test, imgsz=640, batch=16
2025-12-07 19:52:17 I [helper_logger:252] - 모델 검증 시작: self.model_type=yolo_pt
2025-12-07 19:52:17 I [helper_logger:253] - 모델 검증 시작: self.model_config=None
Ultralytics 8.3.235  Python-3.10.18 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i7-1260P)
모델 검증 시작

[YOLOv8m_ONNX_FP32] 검증 중...
2025-12-07 19:52:17 I [helper_logger:251] - 모델 검증 시작: split=test, imgsz=640, batch=16
2025-12-07 19:52:17 I [helper_logger:252] - 모델 검증 시작: self.model_type=yolo_pt
2025-12-07 19:52:17 I [helper_logger:253] - 모델 검증 시작: self.model_config=None
Ultralytics 8.3.235  Python-3.10.18 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i7-1260P)
Loading D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx for

이미지 예측 중:   0%|          | 0/10 [00:00<?, ?it/s]

Loading D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 CPUExecutionProvider
Using ONNX Runtime 1.23.2 CPUExecutionProvider
2025-12-07 19:52:28 I [helper_logger:356] - 예측 완료: 10개 이미지
[YOLOv8m_ONNX_int8_qdq] 예측 중...
2025-12-07 19:52:28 I [helper_logger:321] - 이미지 예측 시작: max_images=None
2025-12-07 19:52:28 I [helper_logger:356] - 예측 완료: 10개 이미지
[YOLOv8m_ONNX_int8_qdq] 예측 중...
2025-12-07 19:52:28 I [helper_logger:321] - 이미지 예측 시작: max_images=None


이미지 예측 중:   0%|          | 0/10 [00:00<?, ?it/s]

2025-12-07 19:52:28 E [helper_logger:537] - 예측 중 오류 (D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test\shiba_inu_129.jpg): [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: images for the following indices
 index: 2 Got: 448 Expected: 640
 Please fix either the inputs/outputs or the model.
2025-12-07 19:52:28 E [helper_logger:537] - 예측 중 오류 (D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test\shiba_inu_13.jpg): [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: images for the following indices
 index: 2 Got: 384 Expected: 640
 Please fix either the inputs/outputs or the model.
2025-12-07 19:52:28 E [helper_logger:537] - 예측 중 오류 (D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test\shiba_inu_13.jpg): [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: images for the following indices
 index: 2 Got: 384 Expected: 640
 Please fix either the inputs/outputs or

In [19]:
# print_dic_tree(results)

In [20]:
# 결과 출력
pipeline.print_summary()

평가 결과: YOLOv8m_ONNX_FP32
mAP50: 0.9950
mAP50-95: 0.9895
정밀도(Precision): 0.9953
재현율(Recall): 1.0000
추론 시간(평균): 475.18ms
--------------------------------------------------------------------------------
테스트된 총 이미지 수: 10
GT 박스가 있는 이미지 수: 10
예측이 있는 이미지 수: 10
이미지당 평균 GT 박스 수: 1.00
이미지당 평균 예측 수: 1.50
평가 결과: YOLOv8m_ONNX_int8_qdq
mAP50: 0.0000
mAP50-95: 0.0000
정밀도(Precision): 0.0000
재현율(Recall): 0.0000
추론 시간(평균): 16.17ms
--------------------------------------------------------------------------------
테스트된 총 이미지 수: 10
GT 박스가 있는 이미지 수: 10
예측이 있는 이미지 수: 0
이미지당 평균 GT 박스 수: 1.00
이미지당 평균 예측 수: 0.00

평가 결과: YOLOv8m_ONNX_FP32
mAP50: 0.9950
mAP50-95: 0.9895
정밀도(Precision): 0.9953
재현율(Recall): 1.0000
추론 시간(평균): 475.18ms
--------------------------------------------------------------------------------
테스트된 총 이미지 수: 10
GT 박스가 있는 이미지 수: 10
예측이 있는 이미지 수: 10
이미지당 평균 GT 박스 수: 1.00
이미지당 평균 예측 수: 1.50
평가 결과: YOLOv8m_ONNX_int8_qdq
mAP50: 0.0000
mAP50-95: 0.0000
정밀도(Precision): 0.0000
재현율(Recall): 0.0000
추론 시간(평균)

In [21]:
# # 비교 DataFrame 생성
# comparison_df = pipeline.get_comparison_dataframe()
# logger.info("모델 비교 결과:")
# display(comparison_df)

### 3.5. ONNX INT8 QDQ 모델 재평가 (QuantizedModelWrapper 사용)

In [22]:
# # yolo_eval 모듈 재로드
# import os, sys
# import importlib
# sys.path.insert(0, os.getcwd())

# # 기존 모듈 완전 제거
# if 'yolo_eval' in sys.modules:
#     del sys.modules['yolo_eval']
    
# # 하위 모듈도 제거
# for key in list(sys.modules.keys()):
#     if key.startswith('yolo_eval.'):
#         del sys.modules[key]

# # 새로 임포트
# from yolo_eval import *

# logger.info("yolo_eval 모듈 재로드 완료 (ONNX Runtime 지원 추가)")

In [23]:
# # 새로운 파이프라인 생성 (ONNX 모델만 테스트)
# pipeline_onnx = YOLOEvaluationPipeline(
#     yaml_path=yaml_path,
#     device=str(__device)
# )

# # FP32 ONNX (비교용)
# yolov8m_onnx_fp32_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp32', 'yolov8m_fp32.onnx', create=False)
# pipeline_onnx.add_model(
#     model_path=yolov8m_onnx_fp32_path,
#     model_name="YOLOv8m_ONNX_FP32",
#     verbose=False,
#     model_type='onnx',
# )

# # INT8 QDQ ONNX (QuantizedModelWrapper 사용)
# yolov8m_onnx_int8_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_int8_qdq', 'yolov8m_int8_qdq.onnx', create=False)
# pipeline_onnx.add_model(
#     model_path=yolov8m_onnx_int8_path,
#     model_name="YOLOv8m_ONNX_int8_qdq_v2",
#     verbose=False,
#     model_type='onnx',
# )

# logger.info("ONNX 모델 파이프라인 구성 완료")

In [24]:
# # ONNX 모델 재평가 실행
# results_onnx = pipeline_onnx.run_full_evaluation(
#     val_kwargs={
#         'project': yolov8m_result_path,
#         'split': 'test',
#         'imgsz': 640,
#         'batch': 16,
#         'conf': 0.001,
#         'iou': 0.45
#     },
#     pred_conf=0.001
# )

# # 결과 출력
# pipeline_onnx.print_summary()

In [25]:
# # 비교 DataFrame 생성
# comparison_onnx_df = pipeline_onnx.get_comparison_dataframe()
# logger.info("ONNX 모델 비교 결과 (QuantizedModelWrapper 사용):")
# display(comparison_onnx_df)

In [26]:
# 개별 이미지 예측 결과 확인
# evaluator = pipeline.evaluators["YOLOv8m_baseline"]
# results_df = evaluator.get_results_dataframe()
# logger.info("개별 이미지 예측 결과 (상위 5개):")
# results_df.head(5)

### 3.4. 양자화 모델 평가 예제 (준비)

In [27]:
# 양자화 모델 평가 예제 (양자화 모델이 있을 때 사용)
"""
# 파이프라인에 여러 모델 추가
pipeline_multi = YOLOEvaluationPipeline(
    yaml_path=yaml_path,
    device=str(__device)
)

# 원본 모델
pipeline_multi.add_model(
    model_path=yolov8m_best_path,
    model_name="YOLOv8m_baseline"
)

# 양자화 모델 (예시)
# quantized_model_path = my_driver_path('modeling', 'model', 'modeling16', 
#                                       'yolov8m_quantized', 'weights', 'best.pt')
# pipeline_multi.add_model(
#     model_path=quantized_model_path,
#     model_name="YOLOv8m_int8"
# )

# 전체 평가 실행
# results_multi = pipeline_multi.run_full_evaluation(
#     val_kwargs={'split': 'test', 'imgsz': 640, 'batch': 16},
#     pred_max_images=10
# )

# 모델 비교
# comparison_df_multi = pipeline_multi.get_comparison_dataframe()
# print("\n모델 성능 비교:")
# display(comparison_df_multi)

# 결과 저장
# output_path = my_driver_path('modeling', 'evaluation', 'model_comparison')
# pipeline_multi.save_results(output_path)
"""

#print("양자화 모델 평가 준비 완료")

'\n# 파이프라인에 여러 모델 추가\npipeline_multi = YOLOEvaluationPipeline(\n    yaml_path=yaml_path,\n    device=str(__device)\n)\n\n# 원본 모델\npipeline_multi.add_model(\n    model_path=yolov8m_best_path,\n    model_name="YOLOv8m_baseline"\n)\n\n# 양자화 모델 (예시)\n# quantized_model_path = my_driver_path(\'modeling\', \'model\', \'modeling16\', \n#                                       \'yolov8m_quantized\', \'weights\', \'best.pt\')\n# pipeline_multi.add_model(\n#     model_path=quantized_model_path,\n#     model_name="YOLOv8m_int8"\n# )\n\n# 전체 평가 실행\n# results_multi = pipeline_multi.run_full_evaluation(\n#     val_kwargs={\'split\': \'test\', \'imgsz\': 640, \'batch\': 16},\n#     pred_max_images=10\n# )\n\n# 모델 비교\n# comparison_df_multi = pipeline_multi.get_comparison_dataframe()\n# print("\n모델 성능 비교:")\n# display(comparison_df_multi)\n\n# 결과 저장\n# output_path = my_driver_path(\'modeling\', \'evaluation\', \'model_comparison\')\n# pipeline_multi.save_results(output_path)\n'

In [28]:
# (레거시) 기존 방식의 단순 테스트 코드
# 새로운 클래스 기반 평가는 위의 파이프라인을 사용하세요

"""
# 간단한 단일 모델 평가 (레거시)
evaluator_simple = YOLOEvaluator(
    model_path=yolov8m_best_path,
    yaml_path=yaml_path,
    model_name="YOLOv8m_simple_test",
    device=str(__device)
)

# 검증만 실행
metrics_simple = evaluator_simple.validate(
    split='test',
    imgsz=640,
    batch=16,
    conf=0.25,
    iou=0.75
)

metrics_simple.print_summary()
"""
#print("레거시 테스트 코드 (주석 처리됨)")

'\n# 간단한 단일 모델 평가 (레거시)\nevaluator_simple = YOLOEvaluator(\n    model_path=yolov8m_best_path,\n    yaml_path=yaml_path,\n    model_name="YOLOv8m_simple_test",\n    device=str(__device)\n)\n\n# 검증만 실행\nmetrics_simple = evaluator_simple.validate(\n    split=\'test\',\n    imgsz=640,\n    batch=16,\n    conf=0.25,\n    iou=0.75\n)\n\nmetrics_simple.print_summary()\n'